# Automatic circuit / head study: results (interactive)

Plots for `python -m elements.experiments.autostudy` (text report: `docs/findings_autostudy.md`).
All figures are Plotly: hover for values, click a legend entry to hide a line, double-click to
isolate it. Section 6 at the bottom is a free explorer.

Colours everywhere: **blue** quantum readout (hollow / dashed = same circuit without memory),
**orange** its PCA dials, **aqua** random Fourier features of the same width, **yellow** raw input.
The **wide faint grey band** is the best standalone classical model on the raw features, retrained
on the same split and seed (stage K).

In [1]:
# Setup: make the repo importable from notebooks/ (or from the repo root).
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from elements.experiments import autostudy as A
from elements.viz import autostudy as V            # summary / paper tables
from elements.viz import autostudy_plotly as P     # Plotly figures + explorer

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.precision", 3)

D = P.load_all()          # every fit of stages A, B, C, K in one tidy frame
print(f"{len(D)} records")

844 records


In [2]:
# Config: what a top-to-bottom run plots (the explorer in section 6 overrides these).
TASK = "nsl_nov"     # nsl_sup | nsl_nov | unsw_sup | unsw_nov | cic_sup | cic_nov
METRIC = None        # None = the task's primary metric (macro_f1 / macro_det@0.01)
SCHEME = "mean"      # "mean" over label schemes, or one scheme, e.g. "binary"

## 1. The configurations

### What was run

| stage | what | runs |
|---|---|---|
| **A** circuit screen | every circuit below, with the two cheap heads, next to its controls | `runs/auto_A_<task>` |
| **B** head scaling | the circuit A picked, plus its no-memory twin, under heads from linear to a 2-layer MLP / AE | `runs/auto_B_<task>` |
| **C** seeds | the picked circuit and the controls under B's best head, seeds 43 and 44 (seed 42 is B) | `runs/auto_C_<task>_s43`, `_s44` |
| **K** classical | standalone classical models on the raw features, same split and seed (the grey band) | `runs/auto_K_<task>` or earlier runs |

**Pick rule, fixed before the run:** the circuit with the best *quantum* score under the primary
head (`mlp256`, novelty `ae256`); B's best head is the one with the best quantum score (ties to the
smaller). Picks are never tuned on the controls.

### Feature views (the same head is trained on each)

* **quantum** - the reservoir readout. Ion trap: per measurement round, `<Z>` of every input
  qubit and `<Z_i Z_j>` of input pairs, plus the memory qubits at the end. Photonic: the
  C(modes, photons) output probabilities.
* **dials** - the PCA vector that *drove* the reservoir (the tight control: same information
  in, no quantum lift).
* **rff** - random Fourier features of the dials, exactly as wide as the quantum readout
  (a classical random lift of matched width; bandwidth by the median heuristic, untuned).
* **raw** - the scaled input with no bottleneck.
* **no memory** (`_nomem`) - the same ion trap with the memory qubits reset every round:
  isolates what the quantum memory contributes.

### Tasks and heads

In [3]:
P.tasks_table()

,kind,dataset,dedup,row cap,label schemes,primary metric,stage-A heads,stage-B heads
task,,,,,,,,
nsl_sup,supervised,nslkdd,False,all,"families6, all23",macro_f1,"lbfgs, mlp256","lbfgs, mlp32, mlp128, mlp512, mlp2048, mlp512x512"
nsl_nov,novelty,nslkdd,False,all,novelty (normal-only),macro_det@0.01,"pca32, ae256-b32","pca32, ae64-b32, ae256-b32, ae1024-b32, ae512x..."
unsw_sup,supervised,nf_unsw_nb15_v2,True,all,"binary, attack10",macro_f1,"lbfgs, mlp256","lbfgs, mlp32, mlp128, mlp512, mlp2048, mlp512x512"
unsw_nov,novelty,nf_unsw_nb15_v2,True,all,novelty (normal-only),macro_det@0.01,"pca32, ae256-b32","pca32, ae64-b32, ae256-b32, ae1024-b32, ae512x..."
cic_sup,supervised,nf_cse_cic_ids2018_v2,True,200000,"binary, family7",macro_f1,"lbfgs, mlp256","lbfgs, mlp32, mlp128, mlp512, mlp2048, mlp512x512"
cic_nov,novelty,nf_cse_cic_ids2018_v2,True,200000,novelty (normal-only),macro_det@0.01,"pca32, ae256-b32","pca32, ae64-b32, ae256-b32, ae1024-b32, ae512x..."


### Circuits (stage A)

One knob moved at a time from the baseline 4+4 all-to-all L8 dt 0.3 ion trap. *encoding slots*
= numbers the circuit can take in (2 angles x input qubits x rounds); a dataset narrower than
that is fed whole, a wider one is cut by PCA. *readout* = width of the quantum feature vector.

In [4]:
P.circuits_table()

,type,input qubits,memory qubits,qubits,rounds L,dt,coupling,J scale,memory,encoding slots,readout,vs baseline
circuit,,,,,,,,,,,,
it4+4_a2a_L8_dt0.3,trapped ion,4.0,4.0,8.0,8.0,0.30,all_to_all,1.0,True,64,84,baseline
it4+4_a2a_L8_dt0.15,trapped ion,4.0,4.0,8.0,8.0,0.15,all_to_all,1.0,True,64,84,dt=0.15
it4+4_a2a_L8_dt0.6,trapped ion,4.0,4.0,8.0,8.0,0.60,all_to_all,1.0,True,64,84,dt=0.6
it4+4_a2a_L8_dt0.3_J2,trapped ion,4.0,4.0,8.0,8.0,0.30,all_to_all,2.0,True,64,84,j_scale=2.0
it4+4_ring_L8_dt0.3,trapped ion,4.0,4.0,8.0,8.0,0.30,ring,1.0,True,64,84,topology=ring
it4+2_a2a_L8_dt0.3,trapped ion,4.0,2.0,6.0,8.0,0.30,all_to_all,1.0,True,64,82,n_mem=2
it4+6_a2a_L8_dt0.3,trapped ion,4.0,6.0,10.0,8.0,0.30,all_to_all,1.0,True,64,86,n_mem=6
it6+4_a2a_L8_dt0.3,trapped ion,6.0,4.0,10.0,8.0,0.30,all_to_all,1.0,True,96,172,n_in=6
it2+6_a2a_L16_dt0.3,trapped ion,2.0,6.0,8.0,16.0,0.30,all_to_all,1.0,True,64,54,"n_in=2, n_mem=6, steps=16"


### Classical reference models (stage K)

Supervised: logistic regression, kNN-5, random forest 300, Extra Trees 300, XGBoost 300, MLP 512x512.
Zero-day (trained on normal traffic only): IsolationForest, one-class SVM, LOF, kNN distance,
Gaussian mixture, autoencoder 512. The band shows the best of them on the metric being plotted.

## 2. Summary: picked circuit vs controls vs classical (3 seeds)

**Note (CIC-IDS2018 supervised).** The J x2 circuit (`it4+4_a2a_L8_dt0.3_J2`) was first lost to a
tag collision and re-run later: it then won stage A on this task (0.810 vs 0.790), but its stages
B/C were paused before they finished. Stages B/C for this task therefore show the runner-up,
`it4+4_a2a_L16_dt0.3`, which has all three seeds. J x2 is in the stage-A screens of every task.

In [5]:
V.summary_table()

,circuit,head,quantum,best control,quantum - control,quantum - no memory,classical best,classical model,seeds
task,,,,,,,,,
nsl_sup,it4+4_a2a_L16_dt0.3,mlp512x512,0.851,0.855,-4.674e-03,0.004,0.918,xgb300,3
nsl_nov,it6+4_a2a_L8_dt0.3,ae1024-b32,0.795,0.747,4.849e-02,0.003,0.769,ae512-b32,3
unsw_sup,3p20m,mlp512x512,0.718,0.718,8.657e-04,NaN,0.738,xgb300,3
unsw_nov,3p12m,ae256-b32,0.367,0.374,-6.800e-03,NaN,0.358,knndist5,3
cic_sup,it4+4_a2a_L16_dt0.3,mlp2048,0.812,0.807,5.577e-03,0.012,0.835,xgb300,3
cic_nov,it4+4_a2a_L16_dt0.3,ae1024-b32,0.803,0.787,1.654e-02,0.030,0.788,knndist5,3


In [6]:
P.fig_seeds(D, metric=METRIC, scheme=SCHEME)

## 3. Which circuit? (stage A, seed 42)

One row per circuit, sorted by the quantum score. Change `TASK` in the config cell, or use the explorer.

In [7]:
P.fig_screen(D, TASK, metric=METRIC, scheme=SCHEME)

## 4. How big must the classical head be? (stage B, seed 42)

In [8]:
P.fig_heads(D, metric=METRIC, scheme=SCHEME)

In [9]:
# smallest head within 1% of each view's best, and its width relative to the view's input
for task in A.TASKS:
    print(task); display(A.saturation(task).set_index("view"))

nsl_sup


,input_width,best,best_head,smallest_99,head_params,width_over_input
view,,,,,,
it4+4_a2a_L16_dt0.3,164,0.867,mlp512x512,mlp512x512,350727,3.122
dials121,121,0.865,mlp512,mlp512,66055,4.231
it4+4_a2a_L16_dt0.3_nomem,164,0.862,mlp2048,mlp2048,352263,12.488
rff164_121m,164,0.859,mlp2048,mlp2048,352263,12.488
raw,121,0.832,mlp512x512,mlp512x512,328711,4.231


nsl_nov


,input_width,best,best_head,smallest_99,head_params,width_over_input
view,,,,,,
it6+4_a2a_L8_dt0.3,172,0.805,ae1024-b32,ae1024-b32,420044,5.953
it6+4_a2a_L8_dt0.3_nomem,172,0.803,ae512x512-b32,ae1024-b32,420044,5.953
dials96,96,0.762,ae1024-b32,ae1024-b32,264320,10.667
raw,122,0.758,ae512x512-b32,ae512x512-b32,684186,4.197
rff172_96m,172,0.598,ae1024-b32,ae1024-b32,420044,5.953


unsw_sup


...

,input_width,best,best_head,smallest_99,head_params,width_over_input
view,,,,,,
3p12m,220,0.396,ae256-b32,ae256-b32,129788,1.164
rff220_12m,220,0.366,ae256-b32,ae256-b32,129788,1.164
raw,37,0.327,ae512x512-b32,ae512x512-b32,576561,13.838
dials12,12,0.286,ae512x512-b32,ae512x512-b32,542736,42.667


cic_sup


,input_width,best,best_head,smallest_99,head_params,width_over_input
view,,,,,,
dials36,36,0.795,mlp512x512,mlp128,4994,3.556
it4+4_a2a_L16_dt0.3,164,0.794,mlp2048,mlp2048,342018,12.488
it4+4_a2a_L16_dt0.3_nomem,164,0.784,mlp2048,mlp32,5346,0.195
raw,36,0.779,mlp512x512,mlp2048,79874,56.889
rff164_36m,164,0.758,mlp2048,mlp128,21378,0.780


cic_nov


,input_width,best,best_head,smallest_99,head_params,width_over_input
view,,,,,,
it4+4_a2a_L16_dt0.3,164,0.826,ae1024-b32,ae1024-b32,403652,6.244
dials37,37,0.823,ae1024-b32,ae1024-b32,102449,27.676
raw,37,0.765,ae512x512-b32,ae512x512-b32,576561,13.838
it4+4_a2a_L16_dt0.3_nomem,164,0.745,ae1024-b32,ae1024-b32,403652,6.244
rff164_37m,164,0.660,ae512x512-b32,ae512x512-b32,727236,3.122


## 5. Sanity check: the paper's numbers, reproduced

Extra Trees on NF-UNSW-NB15-v2 under the paper's protocol (random 70/30, no dedup) and under dedup.

In [10]:
V.paper_check()

,accuracy,attack F1,detection rate,false alarm rate,weighted F1 (10 classes)
Sarhan et al. 2022,0.997,0.970,0.971,0.002,0.99
"ours, paper protocol",0.997,0.968,0.973,0.002,0.99
"ours, dedup",0.973,0.922,0.931,0.018,0.94


## 6. 8-qubit vs 20-qubit ion traps (earlier runs, NSL-KDD, seed 42)

The study above screens 6-10-qubit circuits (most are **8 qubits**: 4 input + 4 memory).
The 20-qubit ring is too slow for it (~0.18 s per row with 8 quantum trajectories), so it was
run separately, on NSL-KDD only:

| run | data | quantum views |
|---|---|---|
| `iontrap_supervised` / `iontrap_novelty` | full NSL-KDD | 8q 4+4 L8 (with / without memory), 8q 4+4 L16, photonic 3p12m |
| `iontrap20_supervised` / `iontrap20_novelty` | 20k-row stratified subsample | 8q 4+4 L8 / L16 (with / without memory), **20q ring 8+12 L16 dt 0.2, 8 trajectories (with / without memory)**, photonic 3p12m |

Every quantum view sits next to its dials / RFF / raw controls under the same heads (filled
marker = MLP / AE head, open = linear / PCA). The grey band is drawn only on the full-data runs:
the 20k-row subsample has no classical run of its own, so compare 8q and 20q there directly.
*(on all23, classes with 1-2 test rows move macro-F1 by several points: read families6 first.)*

In [11]:
R = P.load_ion_runs()   # the four earlier ion-trap runs
P.fig_qubits(R, "iontrap20_supervised")

In [12]:
P.fig_qubits(R, "iontrap20_novelty")

In [13]:
P.fig_qubits(R, "iontrap_supervised")

In [14]:
P.fig_qubits(R, "iontrap_novelty")

### The original 20-qubit script (`reservoir-computing/ion_traps/qrc_iontrap_20q_ring.py`)

Run on the GPU backend: 200 train / 100 test / 50 held-out smurf rows, 100 shots per circuit,
the script's own logistic readout (its C is chosen on test AUC, so both columns are optimistic)
and its ring bug (the wrap-around bond has J = 0). Too few rows for a firm conclusion: the
zero-day column has ~50 normal test rows, so detection at 1% FA is a one-row threshold.

In [15]:
P.script20_table()

,supervised AUC,zero-day AUC,detection @1% FA
"quantum (20q ring, 596 features)",0.782,0.993,1.0
classical (same 122 PCA comps),0.937,1.000,1.0


## 7. Explorer: choose what to plot

* **View** - *Circuit screen* (stage A, one row per circuit with its controls), *Head size*
  (score against head; `B ·` lines are the picked circuit and controls across all heads, `A ·`
  lines add any stage-A circuit at its two heads), *Seeds* (stage B + C, every seed and the mean), *8q vs 20q* (the earlier ion-trap runs of section 6, pick the run with **Run**).
* **Task / Metric / Scheme** - any metric the runs recorded; `mean` averages the label schemes.
* **Lines** - Ctrl- or Shift-click to pick several; then press **Plot**.

Needs a live kernel (the widgets do not render in a static view).

In [16]:
P.explorer(D, R)